# DAC — Delaware peek

Scope: Delaware Medicare clinicians (adds `org_pac_id`, credential, specialty over NPPES).
Source: `data/sources/provider_registry/dac/DAC_NationalDownloadableFile.csv` (~800 MB, plain CSV).
Reading: chunked pandas, filtered to `State` = DE.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PATH = ROOT / "data/sources/provider_registry/dac/DAC_NationalDownloadableFile.csv"
STATE_COL, STATE = "State", "DE"
sample = pd.read_csv(PATH, nrows=5, dtype=str)
print(sample.shape[1], "columns")
sample.head()

31 columns


,NPI,Ind_PAC_ID,Ind_enrl_ID,Provider Last Name,Provider First Name,Provider Middle Name,suff,gndr,Cred,Med_sch,...,adr_ln_1,adr_ln_2,ln_2_sprs,City/Town,State,ZIP Code,Telephone Number,ind_assgn,grp_assgn,adrs_id
0,1568251114,1153817721,I20260102002504,FULGENCIO,TOSCANIA,Y.,NaN,F,MD,OTHER,...,NaN,NaN,NaN,AGUADA,PR,00602,7875890003,Y,M,PR00602XXXXAGXXXXXXXXXX00
1,1427629856,0446726608,I20260415004193,MORALES RODRIGUEZ,ANA,L,NaN,F,MD,OTHER,...,NaN,NaN,NaN,AGUADILLA,PR,00603,4075437801,Y,M,PR00603XXXXAGXXXXXXXXXX00
2,1477128999,1658845508,I20260701004503,ACEVEDO AQUINO,LUIS,A,NaN,M,MD,OTHER,...,NaN,NaN,NaN,AGUADILLA,PR,00603,7876734392,Y,M,PR00603XXXXAGXXXXXXXXXX00
3,1578448098,8921583808,I20260329000120,TROCHE,MARISABEL,NaN,NaN,U,CSW,OTHER,...,NaN,NaN,NaN,BOQUERON,PR,00622,7873180914,Y,M,PR00622XXXXBOXXXXXXXXXX00
4,1205780111,0042786774,I20260420001601,MIRANDA,ISAMAR,NaN,NaN,F,NaN,OTHER,...,NaN,NaN,NaN,DORADO,PR,00646,7872435794,Y,M,PR00646XXXXDOXXXXXXXXXX00


In [2]:
# Scope to Delaware. Chunked scan (~800 MB).
parts = []
for chunk in pd.read_csv(PATH, dtype=str, chunksize=200_000):
    parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(11894, 31)

In [3]:
# Fan-out, group linkage, specialty mix.
print("rows:", len(de), "| unique NPIs:", de["NPI"].nunique())
print("rows per NPI (fan-out):", round(len(de) / de["NPI"].nunique(), 2))
print("org_pac_id non-null %:", round(de["org_pac_id"].notna().mean() * 100, 1))
de["pri_spec"].value_counts().head(20)

rows: 11894 | unique NPIs: 7192
rows per NPI (fan-out): 1.65
org_pac_id non-null %: 90.4


pri_spec
NURSE PRACTITIONER                               1822
PHYSICIAN ASSISTANT                              1659
INTERNAL MEDICINE                                 741
PHYSICAL THERAPIST IN PRIVATE PRACTICE            722
CERTIFIED REGISTERED NURSE ANESTHETIST (CRNA)     650
DIAGNOSTIC RADIOLOGY                              593
FAMILY PRACTICE                                   571
CARDIOVASCULAR DISEASE (CARDIOLOGY)               471
CLINICAL SOCIAL WORKER                            333
EMERGENCY MEDICINE                                314
ANESTHESIOLOGY                                    275
MENTAL HEALTH COUNSELOR                           271
NEUROLOGY                                         249
CHIROPRACTIC                                      208
PSYCHIATRY                                        187
OPTOMETRY                                         178
CLINICAL PSYCHOLOGIST                             169
OBSTETRICS/GYNECOLOGY                             167
GENERAL SURGERY    

## Notes
- Grain: one row per NPI x enrollment x group x practice location (fan-out > 1 expected).
- `org_pac_id` groups clinicians into a billing entity (Q4); blank when address not tied to a group.
- `pri_spec` is free-text; map to the primary-care scope (HISTORY 001) empirically.